In [11]:
# Read raw files

import numpy as np
import os

# Path to folder
folder_path = '/home/kale-chen/Downloads/MDARuns'

data_names = [
    'Run1_Head_FullPhantom_PMMA_with_Film_center_121_2MeV_10MU_10min_HWTrigOn.rawf',
    'Run2_Head_FullPhantom_PMMA_with_Film_center_121_2MeV_400MU_10min_HWTrigOn.rawf',
    'Run3_Head_1_inch_4mmDorenzo_Collimator_FullPhantom_PMMA_with_Film_center_121_2MeV_1200MU_20min_HWTrigOn.rawf',
    'Run4_Head_1_inch_3mmDorenzo_Collimator_FullPhantom_PMMA_with_Film_center_141_6MeV_600MU_15min_HWTrigOn.rawf',
    'Run5_Head_1_inch_2mmDorenzo_Collimator_FullPhantom_PMMA_with_Film_center_141_6MeV_1200MU_20min_HWTrigOn.rawf',
    'Run6_Head_1_inch_2-4-8mm_Collimator_FullPhantom_PMMA_with_Film_141_6MeV_1200MU_20min_HWTrigOn.rawf',
    'Run7_Head_1_inch_no_Collimator_2-4-8mm_Phantom_PMMA_with_Film_105.2MeV_400MU_10min_HWTrigOn.rawf'
]

def decode_event_word(w: int):
    efine_raw =  w        & 0x3FF
    tfine_raw = (w >> 10) & 0x3FF
    ecoarse   = (w >> 20) & 0x3FF
    tcoarse   = (w >> 30) & 0x3FF
    tac_id    = (w >> 40) & 0x3
    channel   = (w >> 42)  # upper 22 bits
    efine = (efine_raw + 27) % 1024
    tfine = (tfine_raw + 27) % 1024
    return {
        "channelID": channel,
        "tacID": tac_id,
        "tcoarse": tcoarse,
        "ecoarse": ecoarse,
        "tfine": tfine,
        "efine": efine,
    }

In [3]:
data_dir = os.path.join(folder_path, data_names[0])
size = os.path.getsize(data_dir)
print(size)

2561437232


In [ ]:
with open(data_dir, 'rb') as f:
    file_header = f.read(64)
    num = 0
    events = []
    while num < 2:
        header1 = int.from_bytes(f.read(8), byteorder='little')
        header2 = int.from_bytes(f.read(8), byteorder='little')
        frame_id = header1 & ((1 << 36) - 1)  # bits 0-35
        frame_size = (header1 >> 36) & 0x7FFF
        n_events = header2 & 0x7FFF
        frame_lost = (header2 & 0x18000) != 0
        print(frame_id, frame_size, n_events, frame_lost)
        if frame_size != n_events + 2:
            print(f'Warning: size mismatch: {frame_size} != {n_events + 2}')
        payload = f.read(n_events * 8)
        if len(payload) != n_events * 8:
            print(f'Warning: payload length mismatch: {len(payload)} != {n_events * 8}')
            break
        event_words = [int.from_bytes(payload[i:i+8], byteorder='little') for i in range(0, len(payload), 8)]
        num += 1

        


1887718 2 0 False
1887791 25 23 False


In [25]:
n = 11
print(event_words[n])
print(decode_event_word(event_words[n]))

615477066499099803
{'channelID': 139943, 'tacID': 1, 'tcoarse': 134, 'ecoarse': 193, 'tfine': 286, 'efine': 182}


In [ ]:
len(events)